# Product Market Fit Analysis of Air Purifiers in India

# Importing libraries

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

# loading data 

In [3]:

aqi = pd.read_csv("aqi - Copy.csv")

In [5]:
aqi.head()

,date,state,area,number_of_monitoring_stations,prominent_pollutants,aqi_value,air_quality_status
0,20-02-2025,Andaman and Nicobar Islands,Sri Vijaya Puram,1,O3,66,Satisfactory
1,21-02-2025,Andaman and Nicobar Islands,Sri Vijaya Puram,1,PM10,67,Satisfactory
2,22-02-2025,Andaman and Nicobar Islands,Sri Vijaya Puram,1,O3,131,Moderate
3,24-02-2025,Andaman and Nicobar Islands,Sri Vijaya Puram,1,CO,41,Good
4,25-02-2025,Andaman and Nicobar Islands,Sri Vijaya Puram,1,O3,87,Satisfactory


# Renaming columns

In [10]:
aqi = aqi.rename(columns={
    'aqi_value': 'AQI',
    'area': 'city',
    'number_of_monitoring_stations': 'NOMS',
    'air_quality_status': 'Status'
})

In [11]:
aqi.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 235785 entries, 0 to 235784
Data columns (total 7 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   date                  235785 non-null  datetime64[ns]
 1   state                 235785 non-null  object        
 2   city                  235785 non-null  object        
 3   NOMS                  235785 non-null  int64         
 4   prominent_pollutants  235785 non-null  object        
 5   AQI                   235785 non-null  int64         
 6   Status                235785 non-null  object        
dtypes: datetime64[ns](1), int64(2), object(4)
memory usage: 12.6+ MB


# Changing to datetime format

In [7]:
aqi['date'] = pd.to_datetime(aqi['date'])

C:\Users\91798\AppData\Local\Temp\ipykernel_24688\469448408.py:1: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  aqi['date'] = pd.to_datetime(aqi['date'])


# No of Cities and all States 

In [15]:
aqi['state'].unique()


array(['Andaman and Nicobar Islands', 'Andhra Pradesh',
       'Arunachal Pradesh', 'Assam', 'Bihar', 'Chandigarh',
       'Chhattisgarh', 'Delhi', 'Gujarat', 'Haryana', 'Himachal Pradesh',
       'Jammu and Kashmir', 'Jharkhand', 'Karnataka', 'Kerala',
       'Madhya Pradesh', 'Maharashtra', 'Manipur', 'Meghalaya', 'Mizoram',
       'Nagaland', 'Odisha', 'Puducherry', 'Punjab', 'Rajasthan',
       'Sikkim', 'Tamil Nadu', 'Telangana', 'Tripura', 'Uttar Pradesh',
       'Uttarakhand', 'West Bengal'], dtype=object)

In [16]:
aqi['city'].nunique()

291

# Calculating Avg AQI of all cities

In [14]:
city_avg_aqi = aqi.groupby('city')['AQI'].mean().round(0)
city_avg_aqi



city
Agartala         127.0
Agra              84.0
Ahmedabad        114.0
Ahmednagar       122.0
Aizawl            47.0
                 ...  
Virudhunagar      61.0
Visakhapatnam    116.0
Vrindavan        105.0
Yadgir            66.0
Yamunanagar      137.0
Name: AQI, Length: 291, dtype: float64

# Top 5 and Bottom 5 Cities

In [17]:
# Recompute city_avg_aqi from the full 'aqi' DataFrame
city_avg_aqi = aqi.groupby('city')['AQI'].mean().reset_index()

# Now sort and get top/bottom 5
top_5_cities = city_avg_aqi.sort_values('AQI', ascending=False).head(5).round(0)
bottom_5_cities = city_avg_aqi.sort_values('AQI', ascending=True).head(5).round(0)



In [19]:
top_5_cities

,city,AQI
60,Byrnihat,240.0
35,Begusarai,207.0
82,Delhi,206.0
103,Greater Noida,202.0
255,Sri Ganganagar,198.0


In [20]:
bottom_5_cities

,city,AQI
269,Tirunelveli,32.0
167,Madikeri,38.0
61,Chamarajanagar,43.0
204,Palkalaiperur,43.0
263,Thanjavur,45.0


# Top 2 and Bottom 2 Pollutants

In [21]:
aqi['prominent_pollutants'].unique()

array(['O3', 'PM10', 'CO', 'PM2.5', 'NO2', 'SO2', 'PM2.5,NO2', 'CO,NO2',
       'PM10,NO2', 'PM2.5,PM10', 'PM2.5,O3', 'PM10,CO', 'PM10,O3',
       'CO,O3', 'NO2,O3', 'PM2.5,CO,O3', 'O3,PM2.5,PM10', 'PM10,NO2,O3',
       'PM10,PM2.5,NO2', 'PM10,NO2,PM2.5,O3', 'SO2,O3', 'PM10,SO2',
       'PM2.5,SO2', 'CO,SO2', 'PM10,PM2.5,SO2', 'PM2.5,PM10,CO',
       'PM2.5,CO,SO2', 'PM10,SO2,O3', 'SO3,CO,O3', 'PM10,CO,SO2',
       'PM2.5,SO2,O3', 'PM10,O3,CO', 'NO2,PM10,CO', 'NO2,CO,O3',
       'PM2.5,CO,NO2', 'NO2,SO2', 'NO2,O3,SO2', 'NH3', 'PM2.5,NH3,O3',
       'PM10,NO2,SO3', 'NO2,SO2,CO', 'PM10,NH3', 'PM2.5,NO2,SO2',
       'PM10,NH3,O3', 'PM2.5,NH3,CO', 'PM10,NH3,CO', 'PM2.5,NH3',
       'NH3,CO,O3', 'O3,NH3'], dtype=object)

In [22]:
from collections import Counter

# Step 1: Create a copy of the relevant columns
state_pollutant = aqi[['state', 'prominent_pollutants']].dropna()

# Step 2: Explode pollutants (split by comma and stack them)
state_pollutant['prominent_pollutants'] = state_pollutant['prominent_pollutants'].str.split(',')
state_pollutant = state_pollutant.explode('prominent_pollutants')

# Step 3: Group by state and count each pollutant
pollutant_counts = state_pollutant.groupby(['state', 'prominent_pollutants']).size().reset_index(name='count')

# Step 4: Get top 2 pollutants per state
top2_pollutants_per_state = (
    pollutant_counts.sort_values(['state', 'count'], ascending=[True, False])
    .groupby('state')
    .head(2)
    .reset_index(drop=True)
)
# Step 5: Get Bottom 2 pollutants per state
bottom2_pollutants_per_state = (
    pollutant_counts.sort_values(['state', 'count'], ascending=[True, False])
    .groupby('state')
    .tail(2)
    .reset_index(drop=True)
)

In [23]:
top2_pollutants_per_state

,state,prominent_pollutants,count
0,Andaman and Nicobar Islands,PM10,36
1,Andaman and Nicobar Islands,CO,10
2,Andhra Pradesh,PM10,3606
3,Andhra Pradesh,PM2.5,2244
4,Arunachal Pradesh,PM2.5,263
...,...,...,...
59,Uttar Pradesh,PM2.5,9054
60,Uttarakhand,PM10,1196
61,Uttarakhand,PM2.5,629
62,West Bengal,PM10,4463


In [24]:
bottom2_pollutants_per_state

,state,prominent_pollutants,count
0,Andaman and Nicobar Islands,O3,5
1,Andaman and Nicobar Islands,PM2.5,2
2,Andhra Pradesh,NO2,259
3,Andhra Pradesh,SO2,11
4,Arunachal Pradesh,SO2,36
...,...,...,...
59,Uttar Pradesh,NH3,1
60,Uttarakhand,O3,359
61,Uttarakhand,CO,148
62,West Bengal,SO2,24


In [25]:
south_states = ['Andhra Pradesh', 'Goa', 'Karnataka', 'Kerala', 'Tamil Nadu', 'Telangana']
north_states =['Delhi','Uttar Pradesh','Haryana','Bihar','Rajasthan','Madhya Pradesh','Jharkhand','Punjab']

In [26]:

top2_south = top2_pollutants_per_state[top2_pollutants_per_state['state'].isin(south_states)]
top2_south

,state,prominent_pollutants,count
2,Andhra Pradesh,PM10,3606
3,Andhra Pradesh,PM2.5,2244
26,Karnataka,PM10,14572
27,Karnataka,CO,3456
28,Kerala,PM10,3538
29,Kerala,PM2.5,1344
52,Tamil Nadu,PM10,7187
53,Tamil Nadu,PM2.5,3016
54,Telangana,PM10,1002
55,Telangana,PM2.5,590


In [27]:
bottom2_south = bottom2_pollutants_per_state[bottom2_pollutants_per_state['state'].isin(south_states)]
bottom2_south

,state,prominent_pollutants,count
2,Andhra Pradesh,NO2,259
3,Andhra Pradesh,SO2,11
26,Karnataka,NH3,34
27,Karnataka,SO3,1
28,Kerala,NH3,10
29,Kerala,SO2,7
52,Tamil Nadu,NO2,500
53,Tamil Nadu,NH3,11
54,Telangana,O3,152
55,Telangana,NO2,119


In [28]:
top2_north = top2_pollutants_per_state[top2_pollutants_per_state['state'].isin(north_states)]
bottom2_north = bottom2_pollutants_per_state[bottom2_pollutants_per_state['state'].isin(north_states)]

In [29]:
top2_north

,state,prominent_pollutants,count
8,Bihar,PM10,10041
9,Bihar,PM2.5,9715
14,Delhi,PM2.5,795
15,Delhi,PM10,715
18,Haryana,PM2.5,11089
19,Haryana,PM10,9146
24,Jharkhand,PM10,323
25,Jharkhand,CO,180
30,Madhya Pradesh,PM10,9855
31,Madhya Pradesh,PM2.5,3750


In [30]:
bottom2_north

,state,prominent_pollutants,count
8,Bihar,NO2,614
9,Bihar,SO2,78
14,Delhi,CO,105
15,Delhi,NO2,67
18,Haryana,SO2,167
19,Haryana,NH3,2
24,Jharkhand,SO2,14
25,Jharkhand,O3,11
30,Madhya Pradesh,SO2,312
31,Madhya Pradesh,SO3,3


# Does AQI vary on Weekends and Weekdays?

In [31]:

# --- Step 1: Filter for the last 1 year of data ---
latest_date = aqi['date'].max()
one_year_ago = latest_date - pd.DateOffset(years=1)
aqi_last_year = aqi[aqi['date'] >= one_year_ago]

# --- Step 2: Filter only for selected metro cities ---
metro_cities = ['Delhi', 'Mumbai', 'Chennai', 'Kolkata', 'Bengaluru', 'Hyderabad', 'Ahmedabad', 'Pune']
aqi_metros = aqi_last_year[aqi_last_year['city'].isin(metro_cities)]

# --- Step 3: Add a column for Day of Week and Weekend Flag ---
aqi_metros['dayofweek'] = aqi_metros['date'].dt.dayofweek  # Monday=0, Sunday=6
aqi_metros['is_weekend'] = aqi_metros['dayofweek'].isin([5, 6])  # Saturday & Sunday as weekend

# --- Step 4: Calculate average AQI for weekends and weekdays per city ---
city_weekend_comparison = (
    aqi_metros.groupby(['city', 'is_weekend'])['AQI']
    .mean()
    .reset_index()
    .pivot(index='city', columns='is_weekend', values='AQI')
    .rename(columns={False: 'Weekday_AQI', True: 'Weekend_AQI'})
    
)

# --- Step 5: Add a column showing the difference ---
city_weekend_comparison['AQI_Change'] = city_weekend_comparison['Weekend_AQI'] - city_weekend_comparison['Weekday_AQI']


C:\Users\91798\AppData\Local\Temp\ipykernel_24688\939434352.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  aqi_metros['dayofweek'] = aqi_metros['date'].dt.dayofweek  # Monday=0, Sunday=6
C:\Users\91798\AppData\Local\Temp\ipykernel_24688\939434352.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  aqi_metros['is_weekend'] = aqi_metros['dayofweek'].isin([5, 6])  # Saturday & Sunday as weekend


In [32]:
city_weekend_comparison

is_weekend,Weekday_AQI,Weekend_AQI,AQI_Change
city,,,
Ahmedabad,114.716475,116.038462,1.321986
Bengaluru,71.896552,72.384615,0.488064
Chennai,71.245211,68.442308,-2.802903
Delhi,208.697318,198.923077,-9.774241
Hyderabad,77.923372,79.009615,1.086244
Kolkata,91.727969,91.259615,-0.468354
Mumbai,91.049808,92.653846,1.604038
Pune,101.954023,100.846154,-1.107869


# Monthly AQI change for states

In [35]:

# Step 1: Extract full month name (e.g., 'Jan', 'Feb', ...)
aqi['month'] = aqi['date'].dt.strftime('%b')  # Short month names

# Step 2: Group by state and month, calculate mean AQI
monthly_avg = aqi.groupby(['state', 'month'])['AQI'].mean().reset_index()

# Step 3: Define month order to fix sorting issue
month_order = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
               'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
monthly_avg['month'] = pd.Categorical(monthly_avg['month'], categories=month_order, ordered=True)

# Step 4: Pivot to get desired format: states as rows, months as columns
monthly_avg_table = monthly_avg.pivot(index='state', columns='month', values='AQI')

# Step 5: Round to 1 decimal
monthly_avg_table = monthly_avg_table.round(1)


In [36]:
monthly_avg_table

month,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
state,,,,,,,,,,,,
Andaman and Nicobar Islands,NaN,67.3,66.9,46.3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Andhra Pradesh,114.7,94.4,78.6,70.9,69.0,63.2,49.7,55.2,53.0,77.2,95.3,99.1
Arunachal Pradesh,79.2,66.6,71.3,63.9,44.3,34.1,25.5,29.5,33.2,31.3,54.5,75.5
Assam,201.2,172.2,142.2,112.5,88.9,70.0,62.9,68.7,76.4,71.4,104.9,149.5
Bihar,245.6,188.4,147.4,161.4,132.2,123.3,79.3,80.3,79.3,120.3,229.6,251.8
Chandigarh,264.3,160.9,114.0,123.6,148.8,130.4,63.8,68.5,81.6,129.9,208.5,212.5
Chhattisgarh,105.2,88.9,92.1,84.7,77.9,66.3,45.2,49.7,47.9,80.6,95.9,93.1
Delhi,323.9,223.1,171.7,206.8,201.9,166.3,89.0,93.8,105.9,221.0,356.0,320.4
Gujarat,132.0,137.8,120.1,101.0,104.2,84.4,69.6,69.6,80.8,120.7,147.7,144.1


# Grouping months to seasons (summer, winter, monsoon)

In [34]:

# Step 1: Extract month number
aqi['month_num'] = aqi['date'].dt.month

# Step 2: Define a function to map month to season
def get_season(month):
    if month in [7, 8, 9]:
        return 'Monsoon'
    elif month in [3, 4, 5, 6]:
        return 'Summer'
    else:  # months 10, 11, 12, 1, 2
        return 'Winter'

# Step 3: Create a new 'season' column
aqi['season'] = aqi['month_num'].apply(get_season)

# Step 4: Group by state and season, calculate average AQI
seasonal_avg_aqi = (
    aqi.groupby(['state', 'season'])['AQI']
    .mean()
    .reset_index()
    .rename(columns={'AQI': 'avg_aqi'})
    .round(1)
)
seasonal_avg_aqi

,state,season,avg_aqi
0,Andaman and Nicobar Islands,Summer,56.4
1,Andaman and Nicobar Islands,Winter,67.3
2,Andhra Pradesh,Monsoon,52.6
3,Andhra Pradesh,Summer,71.0
4,Andhra Pradesh,Winter,96.1
...,...,...,...
90,Uttarakhand,Summer,90.5
91,Uttarakhand,Winter,106.7
92,West Bengal,Monsoon,55.8
93,West Bengal,Summer,94.3


In [35]:
# Pivot the table: states as rows, seasons as columns
seasonal_avg_pivot = seasonal_avg_aqi.pivot(index='state', columns='season', values='avg_aqi')

# Reorder columns for consistent season order
season_order = ['Summer', 'Monsoon', 'Winter']
seasonal_avg_pivot = seasonal_avg_pivot[season_order]

# Round again just in case
seasonal_avg_pivot = seasonal_avg_pivot.round(1)

seasonal_avg_pivot


season,Summer,Monsoon,Winter
state,,,
Andaman and Nicobar Islands,56.4,NaN,67.3
Andhra Pradesh,71.0,52.6,96.1
Arunachal Pradesh,57.3,28.9,64.2
Assam,107.3,69.2,142.1
Bihar,143.2,79.6,207.8
Chandigarh,128.8,71.2,195.6
Chhattisgarh,81.8,47.6,93.0
Delhi,188.2,96.1,289.5
Gujarat,103.1,73.3,136.7


# Zoning of States

In [36]:
south_zone = ['Andhra Pradesh', 'Goa', 'Karnataka', 'Kerala', 'Tamil Nadu', 'Telangana','Puducherry']
north_zone =['Delhi','Uttar Pradesh','Haryana','Bihar','Rajasthan','Himachal Pradesh','Punjab','Uttarakhand','Chandigarh','Jammu and Kashmir']
central_zone=['Madhya Pradesh','Chhattisgarh']
east_zone=['West Bengal','Jharkhand','Odisha']
west_zone=['Maharashtra','Gujarat']
northeast_zone=['Sikkim','Assam','Nagaland','Arunachal Pradesh','Mizoram','Meghalaya','Manipur','Tripura']



In [37]:

# Step 1: Create state-to-zone mapping dictionary
zone_map = {
    'South': south_zone,
    'North': north_zone,
    'Central': central_zone,
    'East': east_zone,
    'West': west_zone,
    'Northeast': northeast_zone,
    
}

state_to_zone = {}
for zone, states in zone_map.items():
    for state in states:
        state_to_zone[state] = zone

# Step 2: Preprocess 'date' and add 'season' and 'zone' columns
aqi['date'] = pd.to_datetime(aqi['date'])
aqi['month_num'] = aqi['date'].dt.month

def get_season(month):
    if month in [7, 8, 9]:
        return 'Monsoon'
    elif month in [3, 4, 5, 6]:
        return 'Summer'
    else:  # months 10, 11, 12, 1, 2
        return 'Winter'

aqi['season'] = aqi['month_num'].apply(get_season)
aqi['zone'] = aqi['state'].map(state_to_zone)

# Step 3: Group by zone and season, compute average AQI
zone_season_avg = (
    aqi.groupby(['zone', 'season'])['AQI']
    .mean()
    .reset_index()
    .rename(columns={'AQI': 'avg_aqi'})
    .round(1)
)

# Step 4: Pivot to get zones as rows, seasons as columns
zone_season_pivot = zone_season_avg.pivot(index='zone', columns='season', values='avg_aqi')

# Step 5: Reorder seasons (optional)
season_order = ['Summer', 'Monsoon', 'Winter']
zone_season_pivot = zone_season_pivot[season_order]

# Final Output
zone_season_pivot


season,Summer,Monsoon,Winter
zone,,,
Central,100.5,58.4,125.3
East,112.0,61.3,158.5
North,131.1,76.9,173.9
Northeast,93.2,55.7,114.3
South,64.9,50.0,77.2
West,101.5,58.4,135.5


# Top 2 Pollutants in each zone

In [38]:


# Step 1: Define zones (reuse your definitions)
zone_map = {
    'South': ['Andhra Pradesh', 'Goa', 'Karnataka', 'Kerala', 'Tamil Nadu', 'Telangana','Puducherry'],
    'North': ['Delhi','Uttar Pradesh','Haryana','Bihar','Rajasthan','Himachal Pradesh','Punjab','Uttarakhand','Chandigarh','Jammu and Kashmir'],
    'Central': ['Madhya Pradesh','Chhattisgarh'],
    'East': ['West Bengal','Jharkhand','Odisha'],
    'West': ['Maharashtra','Gujarat'],
    'Northeast': ['Sikkim','Assam','Nagaland','Arunachal Pradesh','Mizoram','Meghalaya','Manipur','Tripura']
    
}

# Step 2: Create a state → zone mapping
state_to_zone = {state: zone for zone, states in zone_map.items() for state in states}

# Step 3: Add zone column to the DataFrame
aqi['zone'] = aqi['state'].map(state_to_zone)

# Step 4: Handle multiple pollutants per row (split on ',')
pollutant_zone_exploded = (
    aqi.dropna(subset=['prominent_pollutants', 'zone'])  # drop missing values
       .assign(prominent_pollutants=aqi['prominent_pollutants'].str.split(','))
       .explode('prominent_pollutants')
       .assign(prominent_pollutants=lambda df: df['prominent_pollutants'].str.strip())
)

# Step 5: Count occurrences of each pollutant per zone
pollutant_counts = (
    pollutant_zone_exploded.groupby(['zone', 'prominent_pollutants'])
    .size()
    .reset_index(name='count')
)

# Step 6: Get top 2 pollutants per zone
top2_pollutants_per_zone = (
    pollutant_counts.sort_values(['zone', 'count'], ascending=[True, False])
    .groupby('zone')
    .head(2)
    .reset_index(drop=True)
)

# Display final result
top2_pollutants_per_zone


,zone,prominent_pollutants,count
0,Central,PM10,14037
1,Central,PM2.5,4658
2,East,PM10,10076
3,East,PM2.5,6592
4,North,PM10,58137
5,North,PM2.5,42033
6,Northeast,PM10,4655
7,Northeast,PM2.5,4147
8,South,PM10,30318
9,South,PM2.5,10582


# Seasonal Average AQI

In [108]:
# Group by season and compute average AQI
india_season_avg = (
    aqi.groupby('season')['AQI']
    .mean()
    .reset_index()
    .rename(columns={'AQI': 'avg_aqi'})
    .round(1)
)

# Optional: Order seasons
season_order = ['Summer', 'Monsoon', 'Winter']
india_season_avg['season'] = pd.Categorical(india_season_avg['season'], categories=season_order, ordered=True)
india_season_avg = india_season_avg.sort_values('season').reset_index(drop=True)

# Final output
print(india_season_avg)

    season  avg_aqi
0   Summer    107.5
1  Monsoon     65.3
2   Winter    139.6


# Monthly Avg AQI

In [40]:
# Step 1: Extract month name (e.g., Jan, Feb, ...)
aqi['month'] = aqi['date'].dt.strftime('%b')  # Short month names

# Step 2: Compute average AQI per month
monthly_avg_india = (
    aqi.groupby('month')['AQI']
    .mean()
    .reset_index()
    .rename(columns={'AQI': 'avg_aqi'})
    .round(1)
)

# Step 3: Ensure months are in correct order
month_order = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
               'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
monthly_avg_india['month'] = pd.Categorical(monthly_avg_india['month'], categories=month_order, ordered=True)
monthly_avg_india = monthly_avg_india.sort_values('month').reset_index(drop=True)

# Final output
print(monthly_avg_india)

   month  avg_aqi
0    Jan    152.4
1    Feb    124.9
2    Mar    109.5
3    Apr    114.1
4    May    110.0
5    Jun     93.5
6    Jul     62.8
7    Aug     65.5
8    Sep     67.6
9    Oct    107.3
10   Nov    161.1
11   Dec    150.7


# Bengaluru Daywise AQI Status

In [41]:
# Step 1: Filter for Bengaluru and date range 2025
bengaluru_2025 = aqi[
    (aqi['city'] == 'Bengaluru') &
    (aqi['date'] >= '2025-03-01') &
    (aqi['date'] <= '2025-05-31')
]

# Step 2: Count days per air quality category (Status)
Bengaluru_status_25 = bengaluru_2025['Status'].value_counts().reset_index()
Bengaluru_status_25.columns = ['Air_Quality_Category', 'Number_of_Days']
Bengaluru_status_25

,Air_Quality_Category,Number_of_Days
0,Satisfactory,48
1,Moderate,13


In [42]:
# Step 1: Filter for Bengaluru and date range 2024
bengaluru_2024 = aqi[
    (aqi['city'] == 'Bengaluru') &
    (aqi['date'] >= '2024-01-01') &
    (aqi['date'] <= '2024-12-31')
]

# Step 2: Count days per air quality category (Status)
Bengaluru_status_24 = bengaluru_2024['Status'].value_counts().reset_index()
Bengaluru_status_24.columns = ['Air_Quality_Category', 'Number_of_Days']
Bengaluru_status_24

,Air_Quality_Category,Number_of_Days
0,Satisfactory,253
1,Good,65
2,Moderate,48


In [44]:
# Delhi Daywise AQI Status

In [45]:
# Step 1: Filter for Delhi and date range 2024
delhi_2024 = aqi[
    (aqi['city'] == 'Delhi') &
    (aqi['date'] >= '2024-01-01') &
    (aqi['date'] <= '2024-12-31')
]

# Step 2: Count days per air quality category (Status)
Delhi_status_24 = delhi_2024['Status'].value_counts().reset_index()
Delhi_status_24.columns = ['Air_Quality_Category', 'Number_of_Days']
Delhi_status_24

,Air_Quality_Category,Number_of_Days
0,Moderate,143
1,Very Poor,70
2,Poor,70
3,Satisfactory,66
4,Severe,17


# Importing Vehicle dataset

In [48]:
veh = pd.read_csv("vahan - Copy.csv")

In [49]:
veh.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 64841 entries, 0 to 64840
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   year                64841 non-null  int64 
 1   month               64841 non-null  object
 2   state               64841 non-null  object
 3   vehicle_class       64841 non-null  object
 4   fuel                64841 non-null  object
 5   No_of_Reg_vehicles  64841 non-null  int64 
dtypes: int64(2), object(4)
memory usage: 3.0+ MB


# Calculating EV % Statewise

In [52]:
# Step 1: Define electric vehicle types
ev_types = ['ELECTRIC(BOV)', 'PURE EV', 'STRONG HYBRID EV','PLUG-IN HYBRID EV']

# Step 2: Mark rows as EV or not
veh['is_ev'] = veh['fuel'].isin(ev_types)

# Step 3: Group by state and compute EV and total registrations
statewise_ev_data = veh.groupby('state').agg(
    total_vehicles=('No_of_Reg_vehicles', 'sum'),
    ev_vehicles=('is_ev', lambda x: veh.loc[x.index, 'No_of_Reg_vehicles'][x].sum())
)

# Step 4: Calculate EV percentage
statewise_ev_data['ev_percentage'] = (
    statewise_ev_data['ev_vehicles'] / statewise_ev_data['total_vehicles'] * 100
).round(2)

# Display result
statewise_ev_data.sort_values('ev_percentage', ascending=False)


,total_vehicles,ev_vehicles,ev_percentage
state,,,
Tripura,178931,22631,12.65
Delhi,2086374,245938,11.79
Goa,249402,29374,11.78
Chandigarh,154562,17888,11.57
Assam,1889297,183428,9.71
Kerala,2357315,226600,9.61
Karnataka,5413304,480191,8.87
Uttar Pradesh,10782843,921471,8.55
Maharashtra,8287915,650823,7.85


# Comparing %EV and %AQI change

In [53]:
# Step 1: Define EV types
ev_types = ['ELECTRIC(BOV)', 'PURE EV', 'STRONG HYBRID EV','PLUG-IN HYBRID EV']

# Step 2: EV % increase from 2023 to 2024
veh_ev = veh[veh['fuel'].isin(ev_types) & veh['year'].isin([2023, 2024])]
ev_summary = veh_ev.groupby(['state', 'year'])['No_of_Reg_vehicles'].sum().unstack()
ev_summary['ev_percent_change'] = (
    (ev_summary[2024] - ev_summary[2023]) / ev_summary[2023] * 100
).round(2)

# Step 3: AQI % change from 2023 to 2024
aqi_2023_2024 = aqi[aqi['date'].dt.year.isin([2023, 2024])]
aqi_summary = aqi_2023_2024.groupby([aqi['state'], aqi['date'].dt.year])['AQI'].mean().unstack()
aqi_summary['aqi_percent_change'] = (
    (aqi_summary[2024] - aqi_summary[2023]) / aqi_summary[2023] * 100
).round(2)

# Step 4: Merge both summaries
combined_change = ev_summary[['ev_percent_change']].merge(
    aqi_summary[['aqi_percent_change']],
    left_index=True, right_index=True
).reset_index()

# Step 5: Show result
combined_change.sort_values('aqi_percent_change', ascending=True)


,state,ev_percent_change,aqi_percent_change
24,Sikkim,NaN,-22.92
3,Bihar,28.27,-21.67
25,Tamil Nadu,50.08,-19.33
7,Gujarat,-9.33,-13.13
11,Jharkhand,24.41,-11.42
13,Kerala,14.43,-11.14
0,Andhra Pradesh,80.72,-10.88
20,Odisha,60.01,-9.97
26,Tripura,52.49,-9.93
12,Karnataka,21.70,-8.74


# All Cities having AQI > 100

In [57]:
# Step 1: Filter for year 2024
aqi_2024 = aqi[aqi['date'].dt.year == 2024]

# Step 2: Filter rows where status is 'Good' or 'Satisfactory'
good_satis = aqi_2024[aqi_2024['Status'].isin(['Good', 'Satisfactory'])]

# Step 3: Count such days per city and state
good_satis_counts = (
    good_satis.groupby(['state', 'city'])
    .size()
    .reset_index(name='Good_Satisfactory_Days')
)

# Step 4: Filter cities with less than 100 such days
cities_less_than_100 = good_satis_counts[good_satis_counts['Good_Satisfactory_Days'] < 100]

# Step 5: Display results
cities_less_than_100.sort_values('state')


,state,city,Good_Satisfactory_Days
9,Assam,Byrnihat,46
15,Bihar,Araria,57
18,Bihar,Begusarai,58
22,Bihar,Buxar,87
25,Bihar,Hajipur,93
27,Bihar,Kishanganj,89
32,Bihar,Patna,98
36,Bihar,Samastipur,77
38,Bihar,Siwan,31
47,Chhattisgarh,Tumidih,88


# Categorising based on no of unhealthy days and Avg AQI

In [64]:

# Define unhealthy categories
unhealthy_statuses = ['Moderate','Poor', 'Very Poor', 'Severe']

# Step 1: Calculate average AQI per city
avg_aqi_per_city = aqi.groupby('city')['AQI'].mean().reset_index()
avg_aqi_per_city.rename(columns={'AQI': 'avg_aqi'}, inplace=True)

# Step 2: Count total days and unhealthy days per city
aqi['is_unhealthy'] = aqi['Status'].isin(unhealthy_statuses)

city_day_counts = (
    aqi.groupby('city')
    .agg(
        total_days=('Status', 'count'),
        unhealthy_days=('is_unhealthy', 'sum')
    )
    .reset_index()
)

# Step 3: Merge with average AQI
city_air_quality = pd.merge(avg_aqi_per_city, city_day_counts, on='city')

# Step 4: Calculate % of unhealthy days
city_air_quality['unhealthy_pct'] = (city_air_quality['unhealthy_days'] / city_air_quality['total_days']) * 100

# Step 5: Filter for hotspots
hotspots = city_air_quality[
    (city_air_quality['avg_aqi'] > 100) & 
    (city_air_quality['unhealthy_pct'] > 50)
].copy()

# Step 6: Round avg_aqi and unhealthy_pct
hotspots['avg_aqi'] = hotspots['avg_aqi'].round()
hotspots['unhealthy_pct'] = hotspots['unhealthy_pct'].round(1)

# Optional: Add state info if needed
hotspots = pd.merge(hotspots, aqi[['city', 'state']].drop_duplicates(), on='city', how='left')

# Final output
hotspots = hotspots[['city', 'state', 'avg_aqi', 'unhealthy_pct']].sort_values(by='avg_aqi', ascending=False)


In [66]:
hotspots

,city,state,avg_aqi,unhealthy_pct
36,Byrnihat,Assam,240.0,89.2
20,Begusarai,Bihar,207.0,75.4
45,Delhi,Delhi,206.0,82.8
55,Greater Noida,Uttar Pradesh,202.0,84.7
124,Sri Ganganagar,Rajasthan,198.0,88.3
...,...,...,...,...
46,Dewas,Madhya Pradesh,103.0,50.3
129,Tumakuru,Karnataka,103.0,52.4
111,Ratlam,Madhya Pradesh,103.0,54.0
96,Nanded,Maharashtra,103.0,50.9


# Categorising  Hotspots Based on criteria

1.Severe Hotspot (Avg AQI) > 200 and very poor / severe days > 10% or (unhealthy days) > 75% 
2.Established Hotspot 150 < (Avg AQI) < 200 or 50% < (unhealthy days) < 75% 
3.Emerging Hotspot 100 < (Avg AQI) < 150 or 25% < (unhealthy days) < 50%

In [56]:
import pandas as pd

# Define category mappings
unhealthy_statuses = ['Moderate', 'Poor', 'Very Poor', 'Severe']
very_poor_or_severe = ['Very Poor', 'Severe']

# Step 1: Calculate average AQI per city
avg_aqi_per_city = aqi.groupby('city')['AQI'].mean().reset_index()
avg_aqi_per_city.rename(columns={'AQI': 'avg_aqi'}, inplace=True)

# Step 2: Count total days, unhealthy days, and very poor/severe days per city
aqi['is_unhealthy'] = aqi['Status'].isin(unhealthy_statuses)
aqi['is_vp_or_severe'] = aqi['Status'].isin(very_poor_or_severe)

city_status_counts = (
    aqi.groupby('city')
    .agg(
        total_days=('Status', 'count'),
        unhealthy_days=('is_unhealthy', 'sum'),
        vp_or_severe_days=('is_vp_or_severe', 'sum')
    )
    .reset_index()
)

# Step 3: Merge with average AQI
city_classification = pd.merge(avg_aqi_per_city, city_status_counts, on='city')

# Step 4: Calculate percentages
city_classification['unhealthy_pct'] = (city_classification['unhealthy_days'] / city_classification['total_days']) * 100
city_classification['vp_severe_pct'] = (city_classification['vp_or_severe_days'] / city_classification['total_days']) * 100

# Step 5: Categorize cities based on rules
def classify_hotspot(row):
    if row['avg_aqi'] >= 200 or row['unhealthy_pct'] > 75 and row['vp_severe_pct'] >=10 :
        return 'Severe Hotspot'
    elif 150 < row['avg_aqi'] < 200 or  50 < row['unhealthy_pct'] <= 75:
        return 'Established Hotspot'
    elif 100 < row['avg_aqi'] < 150 or  25 < row['unhealthy_pct'] <= 50 :
        return 'Emerging Hotspot'
    else:
        return 'Not Hotspot'

city_classification['Hotspot_Category'] = city_classification.apply(classify_hotspot, axis=1)

# Step 6: Merge with state info
city_classification = pd.merge(city_classification, aqi[['city', 'state']].drop_duplicates(), on='city', how='left')

# Step 7: Round values
city_classification['avg_aqi'] = city_classification['avg_aqi'].round()
city_classification['unhealthy_pct'] = city_classification['unhealthy_pct'].round(1)
city_classification['vp_severe_pct'] = city_classification['vp_severe_pct'].round(1)

# Step 8: Final Output
final_output = city_classification[['city', 'state', 'avg_aqi', 'unhealthy_pct', 'vp_severe_pct', 'Hotspot_Category']]
final_output = final_output.sort_values(by='Hotspot_Category')

# Step 9: Save as CSV
final_output.to_csv('city_hotspot_classification.csv', index=False)

print("✅ City hotspot classification saved as 'city_hotspot_classification.csv'")


✅ City hotspot classification saved as 'city_hotspot_classification.csv'


In [59]:
hotspot = pd.read_csv("city_hotspot_classification.csv")

In [60]:
hotspot.head()

,city,state,avg_aqi,unhealthy_pct,vp_severe_pct,Hotspot_Category
0,Agartala,Tripura,127.0,48.0,3.2,Emerging Hotspot
1,Dungarpur,Rajasthan,104.0,42.3,0.0,Emerging Hotspot
2,Dindigul,Tamil Nadu,99.0,33.1,4.1,Emerging Hotspot
3,Dhule,Maharashtra,103.0,49.8,0.0,Emerging Hotspot
4,Manguraha,Bihar,102.0,41.9,1.0,Emerging Hotspot
